In [ ]:
# EDA

import os, json, requests
from dotenv import load_dotenv

load_dotenv()
STEAM_KEY = os.environ["STEAM_API_KEY"]
HEADERS = {"User-Agent": "steam-feedback student project (handillon1@gmail.com)"}

r = requests.get(
    "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/",
    params={"appid": 413150},  # Stardew Valley
    headers=HEADERS, timeout=30,
)
r.status_code, r.json()

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd

STARDEW, CS2 = 413150, 730
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIXTURES = ROOT / "tests" / "fixtures"
FIXTURES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

from collector import http  # shared: 1s delay, retries timeouts/429/5xx only

def get(url, **params):
    return http.get(url, params=params)

def save_fixture(name, obj):
    with open(FIXTURES / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [ ]:
# Reviews
REVIEWS = "https://api.steampowered.com/IUserReviewsService/GetAppReviews/v1/"

def redact(review):
    """Copy of a review with author.steamid removed, for display only."""
    return {**review, "author": {k: v for k, v in review["author"].items() if k != "steamid"}}

r = get(REVIEWS, appid=STARDEW, num_per_page=100)
resp = r.json()["response"]
print("top-level keys:", [k for k in resp if k != "reviews"])
print("cursor:", resp["cursor"])
print("query_summary:", resp["query_summary"])
print("day_range_used:", resp.get("day_range_used"), "| total_matching:", resp.get("total_matching"))
redact(resp["reviews"][0])

In [ ]:
# Default order is NOT newest-first, it's "most helpful" within a 30-day window.
ts = pd.to_datetime([x["timestamp_created"] for x in resp["reviews"]], unit="s")
print("monotonically decreasing?", ts.is_monotonic_decreasing)
ts[:5]

In [ ]:
for f in ["recent", 0, 1, 2]:
    rev = get(REVIEWS, appid=STARDEW, num_per_page=100, filter=f, **{"languages[0]": "english"}).json()["response"]["reviews"]
    created = pd.Series([x["timestamp_created"] for x in rev])
    updated = pd.Series([x["timestamp_updated"] for x in rev])
    print(f"filter={f!r:9} created desc={created.is_monotonic_decreasing!s:5}  updated desc={updated.is_monotonic_decreasing!s:5}  "
          f"newest={pd.to_datetime(created.max(), unit='s')}")

In [ ]:
from collections import Counter
for params in ({}, {"language": "schinese"}, {"languages": "schinese"}, {"languages[0]": "schinese"},
               {"languages[0]": "english"}, {"languages[0]": "all"}):
    p = get(REVIEWS, appid=CS2, num_per_page=100, filter=1, **params).json()["response"]
    langs = Counter(x["language"] for x in p["reviews"])
    print(f"{str(params):28} total_reviews={p['query_summary']['total_reviews']:>9,}  {dict(langs.most_common(3))}")

In [ ]:
for appid in (STARDEW, CS2, 2767030):  # 2767030 = Marvel Rivals
    totals = {pt: get(REVIEWS, appid=appid, num_per_page=1, filter=1, purchase_type=pt,
                      **{"languages[0]": "english"}).json()["response"]["query_summary"]["total_reviews"]
              for pt in (0, 1, 2)}
    print(appid, {k: f"{v:,}" for k, v in totals.items()}, f"default hides {1 - totals[0] / max(totals[1], 1):.0%}")

In [ ]:
REVIEW_PARAMS = {"num_per_page": 100, "filter": 1, "languages[0]": "english", "purchase_type": 1}

def fetch_page(appid, cursor="*"):
    return get(REVIEWS, appid=appid, cursor=cursor, **REVIEW_PARAMS).json()["response"]

def scrub(page, keep=3, max_text=300):
    """Fixture-safe copy: a few reviews, fake steamids, shortened text."""
    page = json.loads(json.dumps(page))
    if "reviews" in page:
        page["reviews"] = page["reviews"][:keep]
        for i, rev in enumerate(page["reviews"]):
            rev["author"]["steamid"] = str(76561190000000000 + i)  # below the real SteamID64 range
            for field in ("review", "developer_response"):
                if field in rev:
                    rev[field] = rev[field][:max_text]
    return page

for appid in (STARDEW, CS2):
    p1 = fetch_page(appid)
    p2 = fetch_page(appid, p1["cursor"])
    both = p1["reviews"] + p2["reviews"]
    ids1 = {x["recommendationid"] for x in p1["reviews"]}
    ids2 = {x["recommendationid"] for x in p2["reviews"]}
    t1 = pd.to_datetime([x["timestamp_created"] for x in p1["reviews"]], unit="s")
    t2 = pd.to_datetime([x["timestamp_created"] for x in p2["reviews"]], unit="s")
    print(f"--- {appid}")
    print(f"page1 {t1.max()} -> {t1.min()}  desc={t1.is_monotonic_decreasing}")
    print(f"page2 {t2.max()} -> {t2.min()}  desc={t2.is_monotonic_decreasing}")
    print(f"page2 starts after page1 ends: {t2.max() <= t1.min()}   duplicate ids: {len(ids1 & ids2)}")
    print("languages:", Counter(x["language"] for x in both))
    print("query_summary on page 2:", p2["query_summary"])
    assert all(x["language"] == "english" for x in both), "language filter isn't working"
    if appid == STARDEW:
        save_fixture("reviews_413150_page1.json", scrub(p1))
        save_fixture("reviews_413150_page2.json", scrub(p2))

In [ ]:
# What does the end of the stream look like? App 8200 has only ~260 reviews.
cursor, seen = "*", set()
for page in range(1, 10):
    p = fetch_page(8200, cursor)
    ids = [x["recommendationid"] for x in p.get("reviews", [])]
    print(f"page {page}: {len(ids)} reviews, has 'reviews' key={'reviews' in p}, cursor unchanged={p['cursor'] == cursor}")
    if not ids:
        save_fixture("reviews_8200_end_of_stream.json", p)
        break
    seen |= set(ids)
    cursor = p["cursor"]
print("total fetched:", len(seen))

In [ ]:
VALHEIM = 892970
p = get(REVIEWS, appid=VALHEIM, num_per_page=100, filter=0, purchase_type=1, **{"languages[0]": "english"}).json()["response"]
replied = [x for x in p["reviews"] if "developer_response" in x]
print(f"{len(replied)} of {len(p['reviews'])} have a developer reply")
save_fixture("reviews_892970_dev_response.json", scrub({**p, "reviews": replied}, keep=1))

# Which keys are on every review, and which only sometimes?
key_counts = Counter(k for x in p["reviews"] for k in x)
author_counts = Counter(k for x in p["reviews"] for k in x["author"])
print("sometimes present:", {k: n for k, n in key_counts.items() if n < len(p["reviews"])})
print("author keys:", dict(author_counts))
{k: replied[0][k] for k in ("developer_response", "timestamp_dev_responded")} if replied else None

In [ ]:
def parse_review(appid, raw):
    """One flat row per review. author.steamid is dropped here and never stored."""
    author = raw.get("author", {})
    return {
        "appid": appid,
        "recommendationid": raw["recommendationid"],
        "language": raw.get("language"),
        "review": raw.get("review", ""),
        "voted_up": raw.get("voted_up"),
        "created_at": pd.to_datetime(raw["timestamp_created"], unit="s"),
        "updated_at": pd.to_datetime(raw.get("timestamp_updated"), unit="s"),
        "votes_up": raw.get("votes_up", 0),
        "votes_funny": raw.get("votes_funny", 0),
        "weighted_vote_score": float(raw.get("weighted_vote_score", 0)),
        "comment_count": raw.get("comment_count", 0),
        "playtime_at_review": author.get("playtime_at_review"),  # minutes; nested under author
        "playtime_forever": author.get("playtime_forever"),
        "author_num_reviews": author.get("num_reviews"),
        "steam_purchase": raw.get("steam_purchase"),
        "received_for_free": raw.get("received_for_free"),
        "written_during_early_access": raw.get("written_during_early_access"),
        "primarily_steam_deck": raw.get("primarily_steam_deck"),
        "developer_response": raw.get("developer_response"),
        "dev_responded_at": pd.to_datetime(raw["timestamp_dev_responded"], unit="s") if "timestamp_dev_responded" in raw else None,
    }

df = pd.DataFrame([parse_review(VALHEIM, x) for x in p["reviews"]])
assert "steamid" not in df.columns
df.drop(columns="review").head()

In [ ]:
def english_reviews_per_day(appid):
    p = fetch_page(appid)
    ts = [x["timestamp_created"] for x in p.get("reviews", [])]
    if not ts:
        return 0.0
    days = (time.time() - min(ts)) / 86400
    return len(ts) / days

for appid in (STARDEW, CS2, 2767030, 8200):
    print(appid, f"{english_reviews_per_day(appid):,.1f} English reviews/day")

In [ ]:
# players
PLAYERS = "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
for appid in (STARDEW, CS2, 99999999):
    r = get(PLAYERS, appid=appid)
    print(appid, r.status_code, r.json())
save_fixture("players_413150.json", get(PLAYERS, appid=STARDEW).json())

In [ ]:
# news
NEWS = "https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/"
for appid in (STARDEW, CS2):
    for extra in ({}, {"feeds": "steam_community_announcements"}):
        news = get(NEWS, appid=appid, count=50, **extra).json()["appnews"]
        df = pd.DataFrame(news["newsitems"])
        df["date"] = pd.to_datetime(df["date"], unit="s")
        print(f"{appid} {extra or ''}: {len(df)} of {news['count']} posts, {df.date.min():%Y-%m-%d} -> {df.date.max():%Y-%m-%d}")
        if appid == STARDEW and not extra:
            save_fixture("news_413150.json", news)
df.columns.tolist()

In [ ]:
news = get(NEWS, appid=STARDEW, count=50, feeds="steam_community_announcements").json()["appnews"]
df = pd.DataFrame(news["newsitems"])
df["date"] = pd.to_datetime(df["date"], unit="s")
df[["date", "title", "feedname", "feed_type", "tags"]].head(12)

In [ ]:
def news_since(appid, since_ts, count=20, **params):
    items, seen, enddate = [], set(), None
    while True:
        page = get(NEWS, appid=appid, count=count, maxlength=1, **params,
                   **({"enddate": enddate} if enddate else {})).json()["appnews"]["newsitems"]
        new = [x for x in page if x["gid"] not in seen]
        print(f"  enddate={enddate}: {len(page)} posts, {len(page) - len(new)} repeats")
        if not new:
            break
        items += new
        seen |= {x["gid"] for x in new}
        enddate = min(x["date"] for x in new)
        if enddate < since_ts:
            break
    return [x for x in items if x["date"] >= since_ts]

YEAR_AGO = time.time() - 365 * 86400
cs2_news = news_since(CS2, YEAR_AGO)
print(len(cs2_news), "CS2 posts in the last 12 months, oldest", pd.to_datetime(min(x["date"] for x in cs2_news), unit="s"))

In [ ]:
import re
PATCH_TITLE = re.compile(r"\b(?:patch|update|hotfix|hot fix|bug ?fix(?:es)?)\b|\bv?\d+\.\d+(?:\.\d+)+\b", re.I)

def classify(items):
    df = pd.DataFrame(items)
    tags = df["tags"] if "tags" in df else pd.Series([None] * len(df))
    df["by_tag"] = tags.apply(lambda t: isinstance(t, list) and "patchnotes" in t)
    df["by_title"] = df["title"].str.contains(PATCH_TITLE)
    df["is_patch"] = df.by_tag | df.by_title
    return df

for appid in (CS2, STARDEW, 892970, 105600):  # 892970 = Valheim, 105600 = Terraria
    df = classify(news_since(appid, YEAR_AGO, count=100, feeds="steam_community_announcements"))
    flag = "  <- zero tagged patches, check how this dev tags" if df.by_tag.sum() == 0 and df.is_patch.sum() > 0 else ""
    print(f"{appid}: {len(df)} posts | tag {df.by_tag.sum()} | title {df.by_title.sum()} | either {df.is_patch.sum()}{flag}")
    print("    title only:", df[df.by_title & ~df.by_tag].title.head(3).tolist())

In [ ]:
# app details
APPDETAILS = "https://store.steampowered.com/api/appdetails"
for appid in (STARDEW, CS2, 99999999):
    j = get(APPDETAILS, appids=appid, cc="us").json()
    v = j[str(appid)]
    if not v["success"]:
        print(appid, j)
        continue
    d = v["data"]
    print(appid, d["name"], "| type:", d["type"], "| is_free:", d["is_free"])
    print("   price_overview:", d.get("price_overview"))
    print("   genres:", [g["description"] for g in d["genres"]])
    save_fixture(f"appdetails_{appid}.json", j)

In [ ]:
# Several apps per call only works with filters=price_overview. How do free games look?
ids = [STARDEW, CS2, 892970, 105600, 2767030]
r = get(APPDETAILS, appids=",".join(map(str, ids)), cc="us", filters="price_overview")
j = r.json()
for appid in ids:
    data = j[str(appid)].get("data")
    print(appid, type(data).__name__, data)
save_fixture("appdetails_prices_multi.json", j)

def price_cents(entry):
    """Final price in US cents, 0 for free games, None if the lookup failed."""
    if not entry.get("success"):
        return None
    data = entry.get("data")
    if not data:  # free games come back as data: [] (an empty list, not a dict)
        return 0
    return data["price_overview"]["final"]

{appid: price_cents(j[str(appid)]) for appid in ids}

In [ ]:
# store page
from collector.store_page import AGE_GATE_COOKIES, StorePageError, fetch_store_page, parse_tags

html = fetch_store_page(STARDEW)
(FIXTURES / "store_page_413150.html").write_text(html, encoding="utf-8")

# Visible tags: <a class="app_tag"> inside div.glance_tags.popular_tags
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
print("visible:", [a.get_text(strip=True) for a in soup.select("div.glance_tags.popular_tags a.app_tag")])

# Better: same tags with vote counts, from the JSON passed to InitAppTagModal(...)
parse_tags(html)[:5]

In [ ]:
# Mature game (Cyberpunk 2077): age gate
r = get(f"https://store.steampowered.com/app/1091500/", cc="us", l="english")
print("no cookies ->", r.history[0].status_code if r.history else None, r.url)

html = fetch_store_page(1091500)  # sends AGE_GATE_COOKIES and checks the final URL is /app/<id>/
print("with cookies ->", [t["name"] for t in parse_tags(html)][:8])

# Any redirect away from /app/<id>/ raises (FiveM 2676230 redirects to the store home page)
try:
    fetch_store_page(2676230)
except StorePageError as e:
    print("raised:", e)

In [ ]:
# app list 
APPLIST = "https://api.steampowered.com/IStoreService/GetAppList/v1/"

r = get(APPLIST, key=STEAM_KEY).json()["response"]
print("default page:", len(r["apps"]), "apps | have_more_results:", r.get("have_more_results"), "| last_appid:", r.get("last_appid"))

p1 = get(APPLIST, key=STEAM_KEY, max_results=100).json()["response"]
p2 = get(APPLIST, key=STEAM_KEY, max_results=100, last_appid=p1["last_appid"]).json()["response"]
print("p1:", p1["apps"][0]["appid"], "->", p1["apps"][-1]["appid"], "| p2:", p2["apps"][0]["appid"], "->", p2["apps"][-1]["appid"])
save_fixture("applist_page1.json", p1)

big = get(APPLIST, key=STEAM_KEY, max_results=100_000).json()["response"]
print("asked for 100k, got:", len(big["apps"]))

dlc = get(APPLIST, key=STEAM_KEY, max_results=3, include_games="false", include_dlc="true").json()["response"]
print("include_games=false, include_dlc=true:", [a["name"] for a in dlc["apps"]])
p1["apps"][:2]